# Provenance-checked knowledge-graph extraction (issue #664)

An LLM that extracts a knowledge graph must stay **grounded in the source**: every entity and relation in a triple should come from text the model actually located, not text it invented. We express that as a *refinement type* with `Requires` and check it **statically**, before the triples are trusted, with no ground-truth labels.

`Requires(op)` on an argument is a precondition on the value's dataflow *provenance*: "you may only pass me a value that `op` produced." Below, `make_triple`'s span arguments carry `Requires(find_span)`, so a span is acceptable only if it came from `find_span`.

> The extraction programs here are **hand-written** to show the check. They stand in for the function an LLM writes under `SynthesizeAndCall` / CodeAdapt; the same `check_grounded` runs on that synthesized program. No API key or model call is needed for this notebook.

In [1]:
import dataclasses
from typing import Annotated

from effectful.internals.runtime import interpreter
from effectful.ops.effects import Requires, check_requires
from effectful.ops.semantics import apply
from effectful.ops.syntax import defdata, defop


@dataclasses.dataclass(frozen=True)
class Span:
    text: str
    start: int
    end: int


@dataclasses.dataclass(frozen=True)
class Triple:
    subject: Span
    relation: str
    object: Span


@defop
def find_span(document: str, query: str) -> Span:
    """Locate `query` in `document`; raises if it is not there verbatim."""
    i = document.find(query)
    if i < 0:
        raise ValueError(f"{query!r} does not occur in the document")
    return Span(query, i, i + len(query))


@defop
def make_triple(
    subject: Annotated[Span, Requires(find_span)],
    relation: str,
    object: Annotated[Span, Requires(find_span)],
) -> Triple:
    return Triple(subject, relation, object)

In [2]:
def check_grounded(extraction):
    """Reify the extraction program to a term (no op body runs, no LLM) and return its
    provenance violations. Empty == every triple is grounded."""
    with interpreter({apply: defdata}):
        term = extraction()
    return check_requires(term)

## A grounded extraction passes

Both endpoints are spans found in the document, so provenance holds and `check_grounded` returns `{}`.

In [3]:
document = "Paris is the capital of France."

def grounded():
    return [
        make_triple(find_span(document, "Paris"), "capital_of", find_span(document, "France"))
    ]

check_grounded(grounded)

{}

## A hallucinated one is caught

Here the object is a span the model **invented** (`Span("Atlantis")`), not one it found. `check_grounded` flags exactly that argument as missing `find_span` provenance -- statically, before the triple is used.

In [4]:
def hallucinated():
    return [
        make_triple(find_span(document, "Paris"), "capital_of", Span("Atlantis", 0, 8))
    ]

violations = check_grounded(hallucinated)
# render readably: operation -> {argument: which provenance is missing}
{
    op.__name__: {arg: sorted(o.__name__ for o in miss) for arg, miss in d.items()}
    for op, d in violations.items()
}

{'make_triple': {'object': ['find_span']}}

## Only run an extraction once its provenance is verified

The grounded program passed the check, so we run it for real and get the concrete triples.

In [5]:
assert not check_grounded(grounded)  # verified grounded
with interpreter({apply: lambda op, *a, **k: op.__default_rule__(*a, **k)}):
    grounded()

[Triple(subject=Span(text='Paris', start=0, end=5), relation='capital_of', object=Span(text='France', start=24, end=30))]

## Where the LLM comes in

Everything above is the **check**. To make it the *LLM* writing the extraction, the model synthesizes `def extract(document): ...` under `SynthesizeAndCall` (see `docs/source/codeadapt.py`), and `check_grounded` runs on that synthesized program before its triples are trusted -- rejecting and (with `RetryLLMHandler`) retrying any extraction that hallucinates a span. That path needs an API key and a small validation seam in the synthesis handler; the provenance check itself is exactly `check_grounded` shown here.